In [594]:
import pandas as pd

In [595]:
events = pd.read_parquet('../data/events.parquet', engine="pyarrow")

In [620]:
teams = events.groupby('team_id').agg({'team':'max'}).reset_index()
teams.to_csv('../results/teams.csv', index=False)

In [596]:
game_ids = events['game_id'].unique()

In [597]:
subset_games = game_ids

In [598]:
# events = events[events['game_id']=='0f51a5c7-6d54-1917-d4a5-9e83a0b4a1ea']
events = events[(events['game_id'].isin(subset_games)) & (events['period']<4)]
events = events.sort_values(by=['game_id', 'sl_event_id'], ascending=True)

In [599]:
attacking_directions = events.copy()
attacking_directions['attack_direction'] = (attacking_directions['x'] * attacking_directions['x_adj'] >= 0).astype(int)
attacking_directions.loc[attacking_directions['attack_direction'] == 0, 'attack_direction'] = -1
attacking_directions = attacking_directions.groupby(['game_id', 'period', 'team_id']).agg({'attack_direction':'mean'}).reset_index()
attacking_directions = attacking_directions.rename(columns={'team_id':'poss_team'})

In [600]:
events_to_exclude = ['check', 'block', 'assist', 'penaltydrawn', 'penalty', 'controlledentryagainst', 'offside', 'goal', 'failedpasslocation',
                     'pslpr', 'pscarry', 'deflection', 'solpr', 'socarry', 'soshot', 'sogoal', 'dumpinagainst', 'pressure']

In [601]:
unique_events       = events[(events['period_time'] != events['period_time'].shift(1)) & (events['period_time'] != events['period_time'].shift(-1))]
unique_events       = unique_events[~unique_events['event_type'].isin(events_to_exclude)]

duplicate_events    = events[(events['outcome'] == 'successful') & ((events['period_time'] == events['period_time'].shift(1)) | (events['period_time'] == events['period_time'].shift(-1)))]
duplicate_events    = duplicate_events[~duplicate_events['event_type'].isin(events_to_exclude)]

In [ ]:
cleaned_df = pd.concat([unique_events, duplicate_events]).sort_values(by=['game_id', 'sl_event_id'], ascending=True)

In [603]:
possessions = cleaned_df.copy()
possessions['new_poss'] = ((possessions['team_id'] != possessions['team_id'].shift(1)) & (possessions['team_id'] == possessions['team_id'].shift(-1))).astype(int)
possessions['end_poss'] = (possessions['new_poss'].shift(-1) == 1).astype(int)
possessions['poss_team'] = None
possessions.loc[possessions['new_poss']==1, 'poss_team'] = possessions['team_id']
possessions['poss_team'].ffill(inplace=True)
possessions.loc[possessions['team_id'].isna(), 'poss_team'] = None

# possessions = cleaned_df[cleaned_df['new_poss']]
# possessions['possession_id'] = possessions.reset_index().index + 1
new_possessions = possessions[possessions['new_poss']==1]
new_possessions['possession_id'] = new_possessions.groupby('game_id').cumcount() + 1
new_possessions = new_possessions.sort_values(by=['game_id', 'sl_event_id'])
new_possessions = new_possessions[['game_id', 'sl_event_id', 'poss_team', 'possession_id']]

# possessions = pd.merge(possessions, new_possessions, on=['game_id', 'sl_event_id'], how='left')
# possessions['possession_id'].ffill(inplace=True)

# possessions = possessions[['game_id', 'sl_event_id', 'possession_id']]

C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\204479557.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_possessions['possession_id'] = new_possessions.groupby('game_id').cumcount() + 1


In [604]:
cleaned_df = pd.merge(cleaned_df, new_possessions, on=['game_id', 'sl_event_id'], how='left')
cleaned_df['possession_id'].ffill(inplace=True)
cleaned_df['poss_team'].ffill(inplace=True)

In [605]:
faceoffs = cleaned_df[cleaned_df['event_type'].isin(['faceoff', 'whistle'])]
faceoffs['ozone_seq'] = -1

C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\2422741585.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  faceoffs['ozone_seq'] = -1


In [606]:
exit_events             = ['carry', 'controlledbreakout', 'dumpout', 'pass']
controlled_exit_events  = ['CONTROLLED EXIT FROM DZ', 'CONTROLLED BREAKOUT']
outlet_pass_events      = ['outlet', 'stretch', 'outletoffboards', 'stretchoffboards']

controlled_exits        = cleaned_df[cleaned_df['description'].isin(controlled_exit_events)]
dumpout_exits           = cleaned_df[(cleaned_df['event_type'] == 'dumpout') & (cleaned_df['outcome']=='successful')]
pass_exits              = cleaned_df[(cleaned_df['event_type'] == 'pass') & (cleaned_df['outcome']=='successful') & (cleaned_df['detail'].isin(outlet_pass_events))]
misc_exits              = cleaned_df[((cleaned_df['x'] < 25) & (cleaned_df['x'] > -25)) & ((cleaned_df['x'].shift(1) >= 25) | (cleaned_df['x'].shift(1) <= -25))]
all_exits               = pd.concat([controlled_exits, dumpout_exits, pass_exits, misc_exits]).sort_values(by=['game_id', 'sl_event_id'], ascending=True)
all_exits['ozone_seq']  = -1

In [607]:
controlled_entries_against = events[events['event_type']=='controlledentryagainst']
controlled_entries_against = controlled_entries_against[['game_id', 'player_id', 'sl_event_id', 'detail']]
controlled_entries_against = controlled_entries_against.rename(columns={'player_id':'defender_id', 'detail':'entry_detail'})
controlled_entries_against['description'] = 'CONTROLLED ENTRY INTO OZ'
controlled_entries_against = controlled_entries_against.sort_values(by='sl_event_id')

In [608]:
entry_events             = ['carry', 'dumpin', 'pass']
controlled_entry_events  = ['CONTROLLED ENTRY INTO OZ']
entry_pass_events        = ['ozentry', 'ozentrystretchoffboards']

controlled_entries       = cleaned_df[cleaned_df['description'].isin(controlled_entry_events)]
dumpin_entries           = cleaned_df[(cleaned_df['event_type'] == 'dumpin') & (cleaned_df['outcome']=='successful')]
# pass_entries             = cleaned_df[(cleaned_df['event_type'] == 'pass') & (cleaned_df['outcome']=='successful') & (cleaned_df['detail'].isin(entry_pass_events))]
all_entries              = pd.concat([controlled_entries, dumpin_entries]).sort_values(by='sl_event_id', ascending=True)
all_entries              = pd.merge_asof(left=all_entries, right=controlled_entries_against, on='sl_event_id', direction="nearest", by=['game_id', 'description'], tolerance=1)
all_entries['ozone_seq'] = 1

In [609]:
ozone_sequences              = cleaned_df[(cleaned_df['x_adj'] >= 25) & (~cleaned_df['event_type'].isin(['faceoff', 'whistle', 'carry', 'dumpin'])) & (~cleaned_df['detail'].isin(entry_pass_events))]
ozone_sequences['ozone_seq'] = 0
ozone_sequences              = pd.concat([ozone_sequences, faceoffs, all_exits, all_entries]).drop_duplicates(subset=['game_id', 'sl_event_id']).reset_index(drop=True)
ozone_sequences              = pd.merge(ozone_sequences, attacking_directions, on=['game_id', 'period', 'poss_team'], how='left')
ozone_sequences              = ozone_sequences.sort_values(by=['game_id', 'sl_event_id'], ascending=True)

ozone_sequences['ozone_seq_detail'] = None
ozone_sequences.loc[ozone_sequences['ozone_seq']==1, 'ozone_seq_detail'] = 'START'
ozone_sequences.loc[(ozone_sequences['ozone_seq']==0) & (ozone_sequences['ozone_seq'].shift(1)==1), 'ozone_seq_detail'] = 'CONT'
ozone_sequences.loc[(ozone_sequences['ozone_seq']==0) & (ozone_sequences['ozone_seq'].shift(1)==-1), 'ozone_seq_detail'] = 'N/A'
ozone_sequences.loc[ozone_sequences['ozone_seq']==-1, 'ozone_seq_detail'] = 'END'
ozone_sequences['ozone_seq_detail'].ffill(inplace=True)

C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\539556782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ozone_sequences['ozone_seq'] = 0


In [610]:
entry_numbers = ozone_sequences[ozone_sequences['ozone_seq_detail'] == 'START'].sort_values(by=['game_id', 'sl_event_id'])
# entry_numbers['oz_seq_num'] = entry_numbers.reset_index().index + 1
entry_numbers['oz_seq_num'] = entry_numbers.groupby('game_id').cumcount() + 1
entry_numbers = entry_numbers[['game_id', 'sl_event_id', 'oz_seq_num']]

In [611]:
ozone = pd.merge(ozone_sequences, entry_numbers, on=['game_id', 'sl_event_id'], how='left').sort_values(by=['game_id', 'sl_event_id'], ascending=True)
ozone.loc[(ozone['ozone_seq_detail'].isin(['N/A', 'END']) & (~ozone['ozone_seq_detail'].shift(1).isin(['START', 'CONT']))), 'oz_seq_num'] = -9
ozone['oz_seq_num'].ffill(inplace=True)

In [612]:
entry_details = ozone[ozone['ozone_seq_detail']=="START"]
entry_details = entry_details[['game_id', 'period', 'sl_event_id', 'player_id', 'team_id', 'opp_team_id', 'event_type', 'x', 'y', 'attack_direction', 'oz_seq_num', 'game_stint', 'defender_id', 'entry_detail']]

In [613]:
ozone_possession = ozone[ozone['oz_seq_num']>0].copy()
ozone_possession['ozone_team'] = None
ozone_possession.loc[ozone_possession['ozone_seq_detail']=='START', 'ozone_team'] = ozone_possession['team_id'].copy()
ozone_possession['ozone_team'].ffill(inplace=True)

# print(ozone_possession.loc[(ozone_possession['oz_seq_num'].isin([1,2,3])) & (ozone_possession['game_id']=='00b0366a-95c6-5250-2dae-e3dd5c4198bc'), ['team_id', 'ozone_team', 'oz_seq_num', 'team', 'event_type']])
ozone_possession = ozone_possession.groupby(['game_id', 'ozone_team', 'team_id', 'oz_seq_num', 'possession_id']).agg({'period_time':lambda x: max(x) - min(x)}).reset_index()
ozone_possession = ozone_possession[ozone_possession['team_id'] == ozone_possession['ozone_team']]

ozone_possession = ozone_possession.groupby(['game_id', 'oz_seq_num']).agg({'possession_id':'count', 'period_time':'sum'}).reset_index()
ozone_possession = ozone_possession.rename(columns={'possession_id':'oz_possessions', 'period_time':'ozpt'})
print(ozone_possession.head())

                                game_id  oz_seq_num  oz_possessions  \
0  00b0366a-95c6-5250-2dae-e3dd5c4198bc         1.0               1   
1  00b0366a-95c6-5250-2dae-e3dd5c4198bc         2.0               2   
2  00b0366a-95c6-5250-2dae-e3dd5c4198bc         3.0               2   
3  00b0366a-95c6-5250-2dae-e3dd5c4198bc         4.0               2   
4  00b0366a-95c6-5250-2dae-e3dd5c4198bc         5.0               1   

          ozpt  
0         0E-9  
1         0E-9  
2  5.790000000  
3  3.370000000  
4         0E-9  


In [614]:
clean_ozone = ozone[ozone['oz_seq_num']>0]

clean_ozone['entry_type'] = ''
clean_ozone.loc[clean_ozone['ozone_seq_detail']=='START', 'entry_type'] = clean_ozone['event_type'].copy()

clean_ozone['ozone_team'] = ''
clean_ozone.loc[clean_ozone['ozone_seq_detail']=='START', 'ozone_team'] = clean_ozone['team_id'].copy()

clean_ozone['entry_player'] = ''
clean_ozone.loc[clean_ozone['ozone_seq_detail']=='START', 'entry_player'] = clean_ozone['player_id'].copy()

clean_ozone['shot'] = 0
clean_ozone.loc[clean_ozone['event_type']=='shot', 'shot'] = 1

clean_ozone['pass'] = 0
clean_ozone.loc[clean_ozone['event_type']=='pass', 'pass'] = 1

clean_ozone['lpr'] = 0
clean_ozone.loc[clean_ozone['event_type']=='lpr', 'lpr'] = 1

clean_ozone = clean_ozone.groupby(['game_id', 'oz_seq_num']).agg({'shot':'sum', 'pass':'sum', 'lpr':'sum', 'sl_xg_all_shots':'sum', 'period_time': lambda x: max(x)-min(x)}).reset_index()
clean_ozone = clean_ozone.rename(columns={'period_time':'duration'})

full_entry_details = pd.merge(clean_ozone, entry_details, on=['game_id', 'oz_seq_num'], how='left')
full_entry_details = pd.merge(full_entry_details, ozone_possession, on=['game_id', 'oz_seq_num'], how='left')
# print(clean_ozone.head())

C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\723305985.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_ozone['entry_type'] = ''
C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\723305985.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_ozone['ozone_team'] = ''
C:\Users\Flodh\AppData\Local\Temp\ipykernel_20564\723305985.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

In [615]:
full_entry_details['entry_type'] = full_entry_details['event_type'] + '_' + full_entry_details['entry_detail']
full_entry_details.loc[full_entry_details['entry_detail'].isna(), 'entry_type'] = full_entry_details.loc[full_entry_details['entry_detail'].isna(), 'event_type']

In [616]:
full_entry_details.to_csv('../results/full_entry_details.csv', index=False)